## 1. Setup

In [ ]:
import os
os.environ["MUJOCO_GL"]="egl"; os.environ["PYOPENGL_PLATFORM"]="egl"

import sys, math, time, gc, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms.functional as TF

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
device = torch.device("cuda:0")

# ── Оптимизации для RTX Ada (tf32 + cudnn autotune) ──
torch.set_float32_matmul_precision('high')  # tf32 для matmul
torch.backends.cudnn.benchmark = True

MODEL_ID    = "HuggingFaceVLA/smolvla_libero"
CRITIC_PATH = "/workspace/out/critic_resnet.pt"

TASK_DESCRIPTION = "pick up the black bowl between the plate and the ramekin and place it on the plate"

TASK_SUITE_NAME       = "libero_spatial"
TASK_ID               = 0
ENV_IMAGE_SIZE        = 256
SEED                  = 42
MAX_STEPS             = 90
NUM_STABILIZATION_STEPS = 10
INIT_STATES_IDS       = [1, 2, 6, 7, 13, 22, 23, 27, 32, 35, 38, 47, 46] # не менять!

torch.manual_seed(SEED); np.random.seed(SEED)

from huggingface_hub import login
login(token="", add_to_git_credential=False)
print("[hf] logged in")


PyTorch: 2.11.0+cu130, CUDA: True


[hf] logged in


## 2. Image preprocessing с кэшем

Загружаем все изображения LIBERO-spatial в numpy кэш, downscale до 96×96 (для скорости critic'а).
Хранятся uint8 → ~50000 × 96 × 96 × 3 × 2 cameras = ~2.6 GB. Влезает в RAM.

In [2]:
from datasets import load_dataset
ds = load_dataset("k1000dai/libero-spatial")["train"]
print(f"Dataset size: {len(ds)}")

# Кэш в исходном разрешении 256×256 uint8 (~20 GB на диск, столько же в RAM).
# downscale до CRITIC_IMG_SIZE делается на лету на GPU вместе с random shift —
# random shift применяется к ПОЛНОМУ разрешению (8 px на 256 = эквивалент DrQ-v2
# pad=4 на working resolution 128, то есть `4 pixels` из App B.1 PA-RL).
# 128×128 (вместо стандартных 84×84) — для сохранения видимости мелких объектов
# LIBERO (bowl, plate, ramekin), которые при 84×84 теряют до 60% детализации.
RAW_IMG_SIZE   = 256
CRITIC_IMG_SIZE = 128  # было 84: для LIBERO с маленькими объектами
IMG_CACHE_PATH = "/workspace/data/img_cache_critic_256.npz"

if os.path.exists(IMG_CACHE_PATH):
    print(f"Загружаем image cache из {IMG_CACHE_PATH}")
    data = np.load(IMG_CACHE_PATH)
    img1_cache = data["img1"]  # uint8 [N, 96, 96, 3]
    img2_cache = data["img2"]
    state_cache = data["state"].astype(np.float32)
    action_cache = data["action"].astype(np.float32)
    episode_cache = data["episode"].astype(np.int64)
    frame_cache = data["frame"].astype(np.int64)
    print(f"Cache loaded: {img1_cache.shape}")
else:
    print("Строим image cache (один раз, ~3-5 мин)…")
    N = len(ds)
    img1_cache = np.zeros((N, RAW_IMG_SIZE, RAW_IMG_SIZE, 3), dtype=np.uint8)
    img2_cache = np.zeros((N, RAW_IMG_SIZE, RAW_IMG_SIZE, 3), dtype=np.uint8)
    state_cache = np.zeros((N, 8), dtype=np.float32)
    action_cache = np.zeros((N, 7), dtype=np.float32)
    episode_cache = np.zeros(N, dtype=np.int64)
    frame_cache = np.zeros(N, dtype=np.int64)
    
    BATCH = 256
    t0 = time.time()
    for s in range(0, N, BATCH):
        e = min(s + BATCH, N)
        rows = ds[s:e]
        for i, (im1, im2, st, ac, ep, fr) in enumerate(zip(
            rows["observation.images.image"],
            rows["observation.images.wrist_image"],
            rows["observation.state"],
            rows["action"],
            rows["episode_index"],
            rows["frame_index"],
        )):
            # Сохраняем в исходном разрешении — без потери информации
            img1_cache[s+i] = np.array(im1, dtype=np.uint8)
            img2_cache[s+i] = np.array(im2, dtype=np.uint8)
            state_cache[s+i] = np.array(st, dtype=np.float32)
            action_cache[s+i] = np.array(ac, dtype=np.float32)[:7]
            episode_cache[s+i] = ep
            frame_cache[s+i] = fr
        if s % (BATCH * 20) == 0:
            print(f"  {s}/{N} ({100*s/N:.0f}%, {(time.time()-t0):.0f}s)")
    
    print(f"Сохраняем cache в {IMG_CACHE_PATH}…")
    os.makedirs(os.path.dirname(IMG_CACHE_PATH), exist_ok=True)
    np.savez(IMG_CACHE_PATH,
             img1=img1_cache, img2=img2_cache,
             state=state_cache, action=action_cache,
             episode=episode_cache, frame=frame_cache)
    print(f"Готово, {(time.time()-t0)/60:.1f} мин")

print(f"\nCache stats:")
print(f"  img1: {img1_cache.shape} {img1_cache.dtype} ({img1_cache.nbytes/1e9:.2f} GB)")
print(f"  img2: {img2_cache.shape}")
print(f"  state: {state_cache.shape} range [{state_cache.min():.2f}, {state_cache.max():.2f}]")
print(f"  action: {action_cache.shape} range [{action_cache.min():.2f}, {action_cache.max():.2f}]")


Resolving data files:   0%|          | 0/69 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/69 [00:00<?, ?it/s]

Dataset size: 52970
Загружаем image cache из /workspace/data/img_cache_critic_256.npz


Cache loaded: (52970, 256, 256, 3)

Cache stats:
  img1: (52970, 256, 256, 3) uint8 (10.41 GB)
  img2: (52970, 256, 256, 3)
  state: (52970, 8) range [-1.80, 3.46]
  action: (52970, 7) range [-1.00, 1.00]


## 3. Нормализация state

Обычно SmolVLA preprocessor нормализует state. Мы делаем то же — нормализуем по mean/std из datasets.

In [3]:
state_mean = state_cache.mean(axis=0)
state_std  = state_cache.std(axis=0) + 1e-6
state_norm_cache = (state_cache - state_mean) / state_std
print(f"State mean: {state_mean}")
print(f"State std:  {state_std}")
print(f"State norm range: [{state_norm_cache.min():.2f}, {state_norm_cache.max():.2f}]")


State mean: [-0.02446256  0.1065296   1.0580484   3.062847   -0.10464039  0.08307312
  0.01995457 -0.0201628 ]
State std:  [0.11014885 0.13784789 0.10442924 0.10451154 0.41121083 0.21767005
 0.0172619  0.01711264]
State norm range: [-5.41, 4.41]


## 3.5. Загрузка rollouts из нескольких cache-файлов

Rollouts собирались в нескольких отдельных запусках с разными уровнями шума и сохранялись в `rollout_cache_v1.npz`, `v2.npz`, `v3.npz`, …. Эта ячейка автоматически находит все такие файлы по glob-маске, загружает их и склеивает в один combined buffer.

При склейке:
- Episode-индексы **переименовываются** (offset по max-индексу предыдущего файла), чтобы не было коллизий между cache-ами.
- Episode success-маски мерджатся в единый dict.
- Frames с одинаковым episode_id остаются в правильном порядке.


In [4]:
import glob

# === Флаг загрузки rollouts ===
# False — обучаем критик только на demo data (быстрее, но q_gap может быть ниже)
# True  — склеиваем demo + rollouts из rollout_cache_v*.npz
USE_ROLLOUTS = True

if USE_ROLLOUTS:
    # Найти все rollout cache файлы (v1.npz, v2.npz, v3.npz, ...)
    ROLLOUT_CACHE_PATTERN = "/workspace/data/rollout_cache_v*.npz"
    cache_files = sorted(glob.glob(ROLLOUT_CACHE_PATTERN))
    print(f"Найдено {len(cache_files)} cache-файлов:")
    for f in cache_files:
        sz = os.path.getsize(f) / 1e9
        print(f"  {f}  ({sz:.2f} GB)")
    
    if len(cache_files) == 0:
        print("⚠ Нет cache-файлов с rollouts. Переключаемся в demo-only режим.")
        USE_ROLLOUTS = False

if USE_ROLLOUTS:
    # ── Загрузка и склейка ──
    all_img1_chunks, all_img2_chunks = [], []
    all_state_chunks, all_action_chunks = [], []
    all_episode_chunks, all_frame_chunks = [], []
    all_ep_success = {}
    
    current_ep_offset = 0  # для переименования episode-id, чтобы не пересекались между файлами
    
    for cache_path in cache_files:
        print(f"\nЗагружаем {cache_path}…")
        d = np.load(cache_path, allow_pickle=True)
        file_img1     = d["img1"]
        file_img2     = d["img2"]
        file_state    = d["state"].astype(np.float32)
        file_action   = d["action"].astype(np.float32)
        file_episode  = d["episode"].astype(np.int64)
        file_frame    = d["frame"].astype(np.int64)
        file_ep_success = dict(d["ep_success"].item())
    
        n_succ = sum(file_ep_success.values())
        n_total = len(file_ep_success)
        n_fail = n_total - n_succ
        print(f"  frames: {len(file_action)}, episodes: {n_total} ({n_succ} ✓ / {n_fail} ✗)")
    
        # Переименование episode-id с offset-ом
        file_episode_remapped = file_episode + current_ep_offset
        file_ep_success_remapped = {int(ep) + current_ep_offset: succ 
                                     for ep, succ in file_ep_success.items()}
    
        all_img1_chunks.append(file_img1)
        all_img2_chunks.append(file_img2)
        all_state_chunks.append(file_state)
        all_action_chunks.append(file_action)
        all_episode_chunks.append(file_episode_remapped)
        all_frame_chunks.append(file_frame)
        all_ep_success.update(file_ep_success_remapped)
    
        max_in_file = int(file_episode.max()) if len(file_episode) > 0 else -1
        current_ep_offset += max_in_file + 1
    
    rollout_img1_np    = np.concatenate(all_img1_chunks, axis=0)
    rollout_img2_np    = np.concatenate(all_img2_chunks, axis=0)
    rollout_state_np   = np.concatenate(all_state_chunks, axis=0)
    rollout_action_np  = np.concatenate(all_action_chunks, axis=0)
    rollout_episode_np = np.concatenate(all_episode_chunks, axis=0)
    rollout_frame_np   = np.concatenate(all_frame_chunks, axis=0)
    
    n_rollout_succ = sum(all_ep_success.values())
    n_rollout_total = len(all_ep_success)
    n_rollout_fail = n_rollout_total - n_rollout_succ
    print(f"\n=== Все rollouts склеены ===")
    print(f"  total frames:    {len(rollout_action_np):,}")
    print(f"  total episodes:  {n_rollout_total}")
    print(f"    ✓ success: {n_rollout_succ} ({n_rollout_succ/n_rollout_total:.0%})")
    print(f"    ✗ failure: {n_rollout_fail} ({n_rollout_fail/n_rollout_total:.0%})")
    
    assert len(set(rollout_episode_np.tolist())) == n_rollout_total, \
        "Episode index collision after remap — что-то пошло не так!"
    print(f"  ✓ нет коллизий episode-id")
    
    # ── Склейка demo + rollouts ──
    combined_img1     = np.concatenate([img1_cache,    rollout_img1_np], axis=0)
    combined_img2     = np.concatenate([img2_cache,    rollout_img2_np], axis=0)
    combined_state    = np.concatenate([state_cache,   rollout_state_np], axis=0)
    combined_action   = np.concatenate([action_cache,  rollout_action_np], axis=0)
    
    demo_max_ep = int(episode_cache.max())
    rollout_episode_for_combined = rollout_episode_np + demo_max_ep + 1
    combined_episode  = np.concatenate([episode_cache, rollout_episode_for_combined], axis=0)
    combined_frame    = np.concatenate([frame_cache,   rollout_frame_np], axis=0)
    
    combined_episode_success = {int(ep): True for ep in np.unique(episode_cache)}
    combined_episode_success.update({int(ep) + demo_max_ep + 1: succ 
                                      for ep, succ in all_ep_success.items()})
    
    combined_state_norm = ((combined_state - state_mean) / state_std).astype(np.float32)
    
    # Sanity: state distribution
    demo_norm = (state_cache - state_mean) / state_std
    rollout_norm = (rollout_state_np - state_mean) / state_std
    print(f"\nState norm distribution (demo-only mean/std, applied to all):")
    print(f"  demo:    mean abs per-dim = {np.abs(demo_norm).mean(0).round(2)}")
    print(f"  rollout: mean abs per-dim = {np.abs(rollout_norm).mean(0).round(2)}")
    print(f"  rollout max abs:           {np.abs(rollout_norm).max(0).round(2)}")
    
    n_succ_eps = sum(combined_episode_success.values())
    n_fail_eps = len(combined_episode_success) - n_succ_eps
    print(f"\nCombined buffer (demo + всех rollouts):")
    print(f"  total frames:   {len(combined_action):,} "
          f"({len(action_cache):,} demo + {len(rollout_action_np):,} rollouts)")
    print(f"  total episodes: {len(combined_episode_success)}")
    print(f"    ✓ success: {n_succ_eps}")
    print(f"    ✗ failure: {n_fail_eps} ({n_fail_eps/len(combined_episode_success):.1%})")
    
    # Освобождаем chunks-память
    del all_img1_chunks, all_img2_chunks, all_state_chunks
    del all_action_chunks, all_episode_chunks, all_frame_chunks
    gc.collect()
else:
    # === Demo-only режим ===
    print("=== DEMO-ONLY режим (USE_ROLLOUTS=False) ===")
    print("Используем только demonstrations без rollouts.")
    
    combined_img1    = img1_cache
    combined_img2    = img2_cache
    combined_state   = state_cache
    combined_action  = action_cache
    combined_episode = episode_cache
    combined_frame   = frame_cache
    
    # Все demo episodes — success (для reward shaping в cell 10)
    combined_episode_success = {int(ep): True for ep in np.unique(episode_cache)}
    
    combined_state_norm = ((combined_state - state_mean) / state_std).astype(np.float32)
    
    print(f"\nBuffer (demo only):")
    print(f"  total frames:   {len(combined_action):,}")
    print(f"  total episodes: {len(combined_episode_success)}")
    print(f"    ✓ success: {len(combined_episode_success)} (все demo)")

print(f"\nGPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")


Найдено 4 cache-файлов:
  /workspace/data/rollout_cache_v0.npz  (3.47 GB)
  /workspace/data/rollout_cache_v1.npz  (3.46 GB)
  /workspace/data/rollout_cache_v2.npz  (3.57 GB)
  /workspace/data/rollout_cache_v3.npz  (3.73 GB)

Загружаем /workspace/data/rollout_cache_v0.npz…


  frames: 8829, episodes: 100 (74 ✓ / 26 ✗)

Загружаем /workspace/data/rollout_cache_v1.npz…


  frames: 8808, episodes: 100 (74 ✓ / 26 ✗)

Загружаем /workspace/data/rollout_cache_v2.npz…


  frames: 9080, episodes: 100 (64 ✓ / 36 ✗)

Загружаем /workspace/data/rollout_cache_v3.npz…


  frames: 9476, episodes: 100 (40 ✓ / 60 ✗)



=== Все rollouts склеены ===
  total frames:    36,193
  total episodes:  400
    ✓ success: 252 (63%)
    ✗ failure: 148 (37%)
  ✓ нет коллизий episode-id



State norm distribution (demo-only mean/std, applied to all):
  demo:    mean abs per-dim = [0.87 0.84 0.87 0.78 0.7  0.79 0.97 0.97]
  rollout: mean abs per-dim = [0.63 0.36 0.85 0.81 0.38 0.6  0.96 0.93]
  rollout max abs:           [1.8  0.88 1.63 4.5  2.34 3.24 1.2  1.21]

Combined buffer (demo + всех rollouts):
  total frames:   89,163 (52,970 demo + 36,193 rollouts)
  total episodes: 832
    ✓ success: 684
    ✗ failure: 148 (17.8%)

GPU memory: 0.00 GB


# Critic v2 — ResNet-18 from scratch (по статье PA-RL)

**Что меняется vs предыдущих попыток:**
- Эмбеддинги от SmolVLA **не используются** для critic'а. Они хороши для policy action prediction, плохи для оценки Q.
- Вместо них **отдельный ResNet-18 + State MLP** — обучается с нуля на TD loss.
- **Twin Q** (2 critics, target = min) против overestimation.
- **Random shift augmentation 4 px** — стандартный приём из DrQ-v2 / PA-RL.

**Architecture (из статьи Appendix B.1):**
```
img1 (256x256x3) → ResNet18(out=512)  ─┐
img2 (256x256x3) → ResNet18(out=512)  ─┼─ concat → MLP(512,512,512) → Q
state (8d)       → MLP(64)            ─┤   с action concat на каждом слое
action (7d)      → ─────────────────────┘
```

**Запуск:**
1. Pretrain critic ~50k шагов на 50k frames LIBERO-spatial
2. Q-select эвал на 13 init_states
3. Сравним с baseline (8/13) и v1 critic (8/13)

## 4. Replay buffer

Храним:
- Все images на CPU (uint8, 2.6 GB)
- На каждом sample берём из CPU и переносим на GPU
- Используем pinned memory для скорости

In [5]:
class ReplayBuffer:
    """Buffer с images на GPU прямо в uint8. ~20 GB на RTX 6000 Ada (48 GB) спокойно."""
    def __init__(self, img1, img2, state_norm, action, reward, done,
                 next_idx, episode_starts, device):
        self.size = len(img1)
        # ── Всё на GPU — один раз, sample затем zero-copy ──
        print(f"  Перенос image cache на GPU…")
        # uint8 на GPU — ~10 GB на cam = 20 GB total
        self.img1 = torch.from_numpy(img1).to(device)        # [N, 256, 256, 3] uint8
        self.img2 = torch.from_numpy(img2).to(device)
        self.state_norm = torch.from_numpy(state_norm).to(device)  # [N, 8] float32
        self.action = torch.from_numpy(action).to(device)
        self.reward = torch.from_numpy(reward).to(device)
        self.done = torch.from_numpy(done).to(device)
        self.next_idx = torch.from_numpy(next_idx).to(device)
        self.device = device
        print(f"  ✓ buffer на GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB used")
    
    def sample(self, batch_size, device):
        # Случайные индексы прямо на GPU — zero-copy fancy indexing
        idx = torch.randint(0, self.size, (batch_size,), device=device)
        next_idx = self.next_idx[idx]
        
        # uint8 → float + permute. Конверсия на GPU быстрая.
        img1c = self.img1[idx].permute(0, 3, 1, 2).float() / 255.0
        img2c = self.img2[idx].permute(0, 3, 1, 2).float() / 255.0
        img1n = self.img1[next_idx].permute(0, 3, 1, 2).float() / 255.0
        img2n = self.img2[next_idx].permute(0, 3, 1, 2).float() / 255.0
        
        return {
            "img1_curr": img1c,
            "img2_curr": img2c,
            "img1_next": img1n,
            "img2_next": img2n,
            "state_curr": self.state_norm[idx],
            "state_next": self.state_norm[next_idx],
            "action": self.action[idx],
            "reward": self.reward[idx],
            "done": self.done[idx],
        }


# Строим transitions из COMBINED кэша (demo + rollouts)
print("Готовим transitions из combined cache…")
N = len(combined_action)

# Sort by (episode, frame)
sort_order = np.lexsort((combined_frame, combined_episode))
img1_sorted       = combined_img1[sort_order]
img2_sorted       = combined_img2[sort_order]
state_norm_sorted = combined_state_norm[sort_order]
action_sorted     = combined_action[sort_order]
episode_sorted    = combined_episode[sort_order]
frame_sorted      = combined_frame[sort_order]

# Boundaries: где меняется episode_index
is_last_in_ep = np.concatenate((episode_sorted[1:] != episode_sorted[:-1], [True]))

# ── Per-episode reward shaping ──
# Successful episode terminal: r = 0   (paper bias = -1, max=0)
# Failed episode terminal:     r = -1  (timeout, no bootstrap)
# Все промежуточные frames:    r = -1
rewards = np.full(N, -1.0, dtype=np.float32)
dones   = is_last_in_ep.astype(np.float32)

# Векторизованная установка terminal-rewards на основе success-mask
last_indices = np.where(is_last_in_ep)[0]
last_episodes = episode_sorted[last_indices]
# Маска успешных эпизодов на позициях last_indices
success_mask = np.array([combined_episode_success.get(int(ep), True) for ep in last_episodes], dtype=bool)
rewards[last_indices[success_mask]] = 0.0  # success terminals → r=0
# Failed terminals остаются -1 (timeout)

# next_idx
next_idx_arr = np.arange(N)
next_idx_arr[~is_last_in_ep] = np.arange(N)[~is_last_in_ep] + 1

episode_starts = np.concatenate(([0], np.where(episode_sorted[1:] != episode_sorted[:-1])[0] + 1))

n_success_terminals = int(success_mask.sum())
n_failure_terminals = int(len(last_indices) - n_success_terminals)
print(f"Total transitions: {N}, episodes: {len(episode_starts)}")
print(f"  ✓ success terminals (r=0):  {n_success_terminals}")
print(f"  ✗ failure terminals (r=-1): {n_failure_terminals}")
print(f"  Avg episode length: {N / len(episode_starts):.1f}")
if n_failure_terminals == 0:
    print("  ⚠ WARNING: failure terminals = 0 — критик не получит action-discriminating signal!")

buffer = ReplayBuffer(
    img1=img1_sorted, img2=img2_sorted, state_norm=state_norm_sorted,
    action=action_sorted, reward=rewards, done=dones,
    next_idx=next_idx_arr, episode_starts=episode_starts,
    device=device,
)
print(f"\nReplay buffer size: {buffer.size}")


Готовим transitions из combined cache…


Total transitions: 89163, episodes: 832
  ✓ success terminals (r=0):  684
  ✗ failure terminals (r=-1): 148
  Avg episode length: 107.2
  Перенос image cache на GPU…


  ✓ buffer на GPU: 35.1 GB used

Replay buffer size: 89163


## 5. ResNet-18 encoder

Используем torchvision ResNet-18. Из коробки выдаёт 512-d feature после adaptive pool.

Init **with random weights** (`pretrained=False`) — статья говорит "trained from scratch".

In [6]:
from torchvision.models import resnet18

class ImageEncoder(nn.Module):
    """ResNet-18 from scratch → 512-d feature.
    
    PA-RL App B.1 default. Pretrained ImageNet попробовали — q_gap получился
    в 5× меньше при том же N_STEPS, откатили.
    """
    def __init__(self, out_dim=512):
        super().__init__()
        self.backbone = resnet18(weights=None)
        self.backbone.fc = nn.Identity()
        self.out_dim = 512
    
    def forward(self, img):
        # img: [B, 3, H, W] в [0, 1]
        return self.backbone(img)


# Sanity test
enc = ImageEncoder().to(device)
test_img = torch.rand(2, 3, CRITIC_IMG_SIZE, CRITIC_IMG_SIZE, device=device)
out = enc(test_img)
print(f"Encoder output shape: {out.shape}")
print(f"Encoder params: {sum(p.numel() for p in enc.parameters())/1e6:.1f}M (from scratch)")
del enc


Encoder output shape: torch.Size([2, 512])
Encoder params: 11.2M (from scratch)


## 6. Random shift augmentation (DrQ-v2 / PA-RL)

Случайно сдвигаем изображение на ±4 пикселя в каждом направлении. Padding = 4 пикселя по краям, потом random crop. Это улучшает Q-learning стабильность по статье.

In [7]:
def random_shift_and_downscale(img, target_size=CRITIC_IMG_SIZE, pad=8):
    """Random shift + downscale через grid_sample на GPU.
    
    Память-эффективная альтернатива fancy indexing — позволяет делать batched
    augmentation на [4B, 3, 256, 256] без OOM.
    
    pad=8 на raw 256 = эквивалент DrQ-v2 standard pad=4 на final 128.
    """
    B, C, H, W = img.shape
    # Continuous shift в [-pad, +pad] pixels, переведённый в normalized grid coords [-1, 1]
    shift_norm = (torch.rand(B, 2, device=img.device) * 2 - 1) * (pad / (H / 2))
    
    # Базовая identity-grid в target resolution
    ys = torch.linspace(-1, 1, target_size, device=img.device)
    xs = torch.linspace(-1, 1, target_size, device=img.device)
    grid_y, grid_x = torch.meshgrid(ys, xs, indexing='ij')
    base_grid = torch.stack([grid_x, grid_y], dim=-1)  # [T, T, 2]
    base_grid = base_grid.unsqueeze(0).expand(B, -1, -1, -1)  # [B, T, T, 2]
    
    grid = base_grid + shift_norm.view(B, 1, 1, 2)
    
    return F.grid_sample(img, grid, mode='bilinear', padding_mode='border', align_corners=True)


def downscale_only(img, target_size=CRITIC_IMG_SIZE):
    """Downscale без random shift — для inference в PA-RL и Q-select эвал."""
    if target_size != img.shape[-1]:
        return F.interpolate(img, size=target_size, mode='bilinear', align_corners=False)
    return img


# Sanity
test = torch.randn(4, 3, RAW_IMG_SIZE, RAW_IMG_SIZE, device=device)
out = random_shift_and_downscale(test, target_size=CRITIC_IMG_SIZE, pad=8)
print(f"Input:  {test.shape}")
print(f"Output: {out.shape}")
print(f"grid_sample-based augmentation: continuous shift ±8px на raw {RAW_IMG_SIZE} → {CRITIC_IMG_SIZE}")


Input:  torch.Size([4, 3, 256, 256])
Output: torch.Size([4, 3, 128, 128])
grid_sample-based augmentation: continuous shift ±8px на raw 256 → 128


## 7. Twin Q + V networks

Архитектура по статье (Kumar et al. 2023): action concatenates на каждом слое MLP.
Twin Q — два независимых critic'а, target = min(Q1, Q2).

In [8]:
class CriticHead(nn.Module):
    """Critic head с action embedding — критично для случая когда obs >> action в scale.
    Без этого action [-1,1] заглушается obs из ResNet с большой нормой."""
    def __init__(self, obs_dim, action_dim=7, hidden=512, action_emb_dim=128):
        super().__init__()
        # Action embedding — поднимаем 7-d action до 128-d с LayerNorm
        self.action_emb = nn.Sequential(
            nn.Linear(action_dim, action_emb_dim),
            nn.LayerNorm(action_emb_dim),
            nn.ReLU(),
            nn.Linear(action_emb_dim, action_emb_dim),
            nn.LayerNorm(action_emb_dim),
            nn.ReLU(),
        )
        # Q head: action_emb concatenates на каждом слое (Kumar et al. 2023)
        self.l1 = nn.Sequential(nn.Linear(obs_dim+action_emb_dim, hidden), nn.LayerNorm(hidden), nn.ReLU())
        self.l2 = nn.Sequential(nn.Linear(hidden+action_emb_dim, hidden), nn.LayerNorm(hidden), nn.ReLU())
        self.l3 = nn.Sequential(nn.Linear(hidden+action_emb_dim, hidden), nn.LayerNorm(hidden), nn.ReLU())
        self.l4 = nn.Linear(hidden+action_emb_dim, 1)
    def forward(self, obs, a):
        a_emb = self.action_emb(a)  # [B, 128] — нормализованная action representation
        x = self.l1(torch.cat([obs, a_emb], -1))
        x = self.l2(torch.cat([x, a_emb], -1))
        x = self.l3(torch.cat([x, a_emb], -1))
        return self.l4(torch.cat([x, a_emb], -1)).squeeze(-1)


class CriticEnsemble(nn.Module):
    """Image encoders + state MLP + 2 critic heads.
    Енкодеры shared между Q1 и Q2.
    """
    def __init__(self, state_dim=8, action_dim=7, hidden=512, state_hidden=64):
        super().__init__()
        self.enc1 = ImageEncoder()  # overhead camera
        self.enc2 = ImageEncoder()  # wrist camera
        self.state_mlp = nn.Sequential(
            nn.Linear(state_dim, state_hidden), nn.LayerNorm(state_hidden), nn.ReLU(),
            nn.Linear(state_hidden, state_hidden),
        )
        # obs_dim = 512 (img1) + 512 (img2) + state_hidden (64) = 1088
        self.obs_dim = 512 + 512 + state_hidden
        # LayerNorm на финальный obs — критично чтобы obs/action были одного scale
        self.obs_norm = nn.LayerNorm(self.obs_dim)
        self.q1 = CriticHead(self.obs_dim, action_dim, hidden)
        self.q2 = CriticHead(self.obs_dim, action_dim, hidden)
    
    def encode(self, img1, img2, state):
        e1 = self.enc1(img1)                          # [B, 512]
        e2 = self.enc2(img2)                          # [B, 512]
        s  = self.state_mlp(state)                    # [B, 64]
        obs = torch.cat([e1, e2, s], dim=-1)          # [B, 1088]
        # ── LayerNorm — выравнивает scale obs с action_emb ──
        # Без этого obs std ~5-10, action_emb std ~1 → action signal заглушается
        obs = self.obs_norm(obs)
        return obs
    
    def forward(self, img1, img2, state, action):
        obs = self.encode(img1, img2, state)
        return self.q1(obs, action), self.q2(obs, action)
    
    def min_q(self, img1, img2, state, action):
        q1, q2 = self.forward(img1, img2, state, action)
        return torch.minimum(q1, q2)


class VNetwork(nn.Module):
    """V(obs) — share encoder через критика? Нет, отдельный.
    Чтобы избежать доп. проходов encoder'а, можем шарить."""
    def __init__(self, obs_dim, hidden=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )
    def forward(self, obs):
        return self.net(obs).squeeze(-1)


critic = CriticEnsemble().to(device)
critic_target = CriticEnsemble().to(device)
critic_target.load_state_dict(critic.state_dict())
for p in critic_target.parameters():
    p.requires_grad_(False)

v_net = VNetwork(critic.obs_dim).to(device)

print(f"Critic params: {sum(p.numel() for p in critic.parameters())/1e6:.1f}M")
print(f"V-net params: {sum(p.numel() for p in v_net.parameters())/1e6:.1f}M")
print(f"Encoder params per ResNet: {sum(p.numel() for p in critic.enc1.parameters())/1e6:.1f}M")


Critic params: 25.0M
V-net params: 1.1M
Encoder params per ResNet: 11.2M


## 8. IQL + Cal-QL training loop

Параметры по статье (Appendix B.1):
- IQL τ = 0.7 для не-AntMaze
- γ = 0.99
- Cal-QL α = 0.005
- Critic LR = 3e-4
- Soft update τ = 0.005

Random shift augmentation на каждом батче.

In [9]:
import torch.nn.functional as F
from torch.amp import autocast

# ── Hyperparameters ──
GAMMA       = 0.99
EXPECTILE   = 0.7
CAL_ALPHA   = 0.005
LR_CRITIC   = 3e-4
LR_V        = 3e-4
TARGET_TAU  = 0.005

critic_optim = torch.optim.Adam(critic.parameters(), lr=LR_CRITIC)
v_optim      = torch.optim.Adam(v_net.parameters(),  lr=LR_V)
print(f"Optimizers: Adam, LR={LR_CRITIC}")

# Vectorized Polyak параметры (cache once)
target_params = list(critic_target.parameters())
online_params = list(critic.parameters())


def expectile_loss(diff, expectile):
    weight = torch.where(diff > 0, expectile, 1 - expectile)
    return weight * (diff ** 2)


def iql_step(batch):
    # ── Batched augmentation [4B, 3, 256, 256] ──
    # grid_sample memory-friendly, OOM не будет
    B = batch["img1_curr"].shape[0]
    all_imgs = torch.cat([batch["img1_curr"], batch["img2_curr"],
                          batch["img1_next"], batch["img2_next"]], dim=0)
    all_aug = random_shift_and_downscale(all_imgs)
    img1c, img2c, img1n, img2n = torch.chunk(all_aug, 4, dim=0)
    
    state_c = batch["state_curr"]
    state_n = batch["state_next"]
    action  = batch["action"]
    reward  = batch["reward"]
    done    = batch["done"]
    
    # ── ENCODE: bf16 autocast (encoder + critic-heads + V-net) ──
    with autocast(device_type='cuda', dtype=torch.bfloat16):
        with torch.no_grad():
            obs_curr_target = critic_target.encode(img1c, img2c, state_c)
            obs_next_target = critic_target.encode(img1n, img2n, state_n)
            q1_target_bf = critic_target.q1(obs_curr_target, action)
            q2_target_bf = critic_target.q2(obs_curr_target, action)
            next_v_bf    = v_net(obs_next_target)
        obs_curr = critic.encode(img1c, img2c, state_c)
        v_val_bf = v_net(obs_curr_target)
        
        q1_bf = critic.q1(obs_curr, action)
        q2_bf = critic.q2(obs_curr, action)
        
        a_random = torch.rand_like(action) * 2 - 1
        q1_rand_bf = critic.q1(obs_curr, a_random)
        q2_rand_bf = critic.q2(obs_curr, a_random)
        
        with torch.no_grad():
            v_ref_bf = v_net(obs_curr.detach())
    
    # ── Losses в fp32 (критично! bf16 quantization съест TD signal) ──
    q1_target = q1_target_bf.float()
    q2_target = q2_target_bf.float()
    q_target_val = torch.minimum(q1_target, q2_target)
    v_val = v_val_bf.float()
    next_v = next_v_bf.float()
    q1 = q1_bf.float()
    q2 = q2_bf.float()
    q1_rand = q1_rand_bf.float()
    q2_rand = q2_rand_bf.float()
    v_ref = v_ref_bf.float()
    
    # 1. V update
    v_loss = expectile_loss(q_target_val - v_val, EXPECTILE).mean()
    v_optim.zero_grad(); v_loss.backward(); v_optim.step()
    
    # 2. Q update — td_loss = СУММА (как в baseline)
    bellman_target = reward + GAMMA * (1.0 - done) * next_v
    td_loss = F.mse_loss(q1, bellman_target) + F.mse_loss(q2, bellman_target)
    
    # Cal-QL
    q_rand = torch.minimum(q1_rand, q2_rand)
    q_demo_avg = (q1 + q2) / 2
    cal_reg = (torch.maximum(q_rand, v_ref) - q_demo_avg.detach()).mean()
    margin_loss = F.relu(q_rand - q_demo_avg + 1.0).mean()
    
    q_loss = td_loss + CAL_ALPHA * cal_reg
    critic_optim.zero_grad(); q_loss.backward(); critic_optim.step()
    
    # Polyak vectorized через _foreach
    with torch.no_grad():
        torch._foreach_mul_(target_params, 1 - TARGET_TAU)
        torch._foreach_add_(target_params, online_params, alpha=TARGET_TAU)
    
    return {
        "td_loss":     td_loss.item(),
        "v_loss":      v_loss.item(),
        "margin_loss": margin_loss.item(),
        "cal_reg":     cal_reg.item(),
        "q_demo":      q_demo_avg.mean().item(),
        "q_random":    q_rand.mean().item(),
        "q_diff":      (q1 - q2).abs().mean().item(),
    }


# ── Training loop ──
N_STEPS = 50000
BATCH_SIZE = 256
LOG_EVERY = 1000

# Если True — загружает checkpoint И продолжает обучение ещё N_STEPS шагов
# Если False — если checkpoint есть, только загружает (skip training)
CONTINUE_TRAINING = True

checkpoint_exists = os.path.exists(CRITIC_PATH)

if checkpoint_exists:
    print(f"Загружаем существующий critic из {CRITIC_PATH}")
    state = torch.load(CRITIC_PATH, map_location=device, weights_only=False)
    critic.load_state_dict(state["critic"])
    critic_target.load_state_dict(state["critic_target"])
    v_net.load_state_dict(state["v_net"])
    print("  OK")

# Условие тренировки: либо checkpoint-а нет (обучение с нуля), либо явно continue
should_train = (not checkpoint_exists) or CONTINUE_TRAINING

if should_train:
    if checkpoint_exists:
        print(f"\n[CONTINUE_TRAINING=True] Дообучаем critic ещё на {N_STEPS} шагах…")
        print(f"  (Adam optimizer state НЕ сохраняется — первые ~500 шагов momentum re-builds)")
    else:
        print(f"\nТренируем critic с нуля на {N_STEPS} шагах…")
    
    t0 = time.time()
    for step in range(1, N_STEPS+1):
        batch = buffer.sample(BATCH_SIZE, device)
        m = iql_step(batch)
        if step % LOG_EVERY == 0:
            elapsed = time.time() - t0
            it_s = step / elapsed
            eta_s = (N_STEPS - step) / it_s
            print(f"step {step:6d}/{N_STEPS} | {it_s:5.1f} it/s | ETA {eta_s/60:4.1f}min "
                  f"| td={m['td_loss']:.4f} | margin={m['margin_loss']:.4f} | cal={m['cal_reg']:.4f} "
                  f"| q_demo={m['q_demo']:.4f} | q_rand={m['q_random']:.4f} "
                  f"| q_gap={m['q_demo']-m['q_random']:+.3f} | q_diff={m['q_diff']:.3f}",
                  flush=True)
    
    torch.save({
        "critic":        critic.state_dict(),
        "critic_target": critic_target.state_dict(),
        "v_net":         v_net.state_dict(),
        "state_mean":    state_mean,
        "state_std":     state_std,
    }, CRITIC_PATH)
    print(f"\nСохранено в {CRITIC_PATH}")

critic.eval()
v_net.eval()


Optimizers: Adam, LR=0.0003

Тренируем critic с нуля на 50000 шагах…


step   1000/50000 |   8.5 it/s | ETA 96.0min | td=0.8605 | margin=1.0114 | cal=0.8343 | q_demo=-6.7337 | q_rand=-6.7224 | q_gap=-0.011 | q_diff=0.023


step   2000/50000 |   8.6 it/s | ETA 92.8min | td=0.1893 | margin=1.0699 | cal=0.8182 | q_demo=-10.9741 | q_rand=-10.9066 | q_gap=-0.067 | q_diff=0.021


step   3000/50000 |   8.7 it/s | ETA 90.2min | td=0.6701 | margin=1.0270 | cal=0.7038 | q_demo=-14.5815 | q_rand=-14.5622 | q_gap=-0.019 | q_diff=0.025


step   4000/50000 |   8.7 it/s | ETA 88.0min | td=1.0167 | margin=1.0138 | cal=0.6793 | q_demo=-17.7842 | q_rand=-17.7886 | q_gap=+0.004 | q_diff=0.024


...

step  48000/50000 |  11.7 it/s | ETA  2.9min | td=1.2118 | margin=0.3360 | cal=0.1760 | q_demo=-37.6045 | q_rand=-39.0564 | q_gap=+1.452 | q_diff=0.095


step  49000/50000 |  11.7 it/s | ETA  1.4min | td=1.7278 | margin=0.3485 | cal=0.1691 | q_demo=-39.8471 | q_rand=-41.4064 | q_gap=+1.559 | q_diff=0.108


step  50000/50000 |  11.7 it/s | ETA  0.0min | td=0.9982 | margin=0.3186 | cal=0.0845 | q_demo=-39.6158 | q_rand=-41.3651 | q_gap=+1.749 | q_diff=0.109



Сохранено в /workspace/out/critic_resnet.pt


VNetwork(
  (net): Sequential(
    (0): Linear(in_features=1088, out_features=512, bias=True)
    (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (2): ReLU()
    (3): Linear(in_features=512, out_features=512, bias=True)
    (4): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (5): ReLU()
    (6): Linear(in_features=512, out_features=512, bias=True)
    (7): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (8): ReLU()
    (9): Linear(in_features=512, out_features=1, bias=True)
  )
)

## 9. LIBERO env init

In [10]:
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy, make_att_2d_masks
from lerobot.policies.factory import make_pre_post_processors

policy = SmolVLAPolicy.from_pretrained(MODEL_ID).to(device).eval()
preprocessor, postprocessor = make_pre_post_processors(
    policy_cfg=policy.config, pretrained_path=MODEL_ID,
    preprocessor_overrides={"device_processor": {"device": str(device)}},
)

import builtins
builtins.input = lambda _: "n"
os.environ["LIBERO_DATA_PATH"] = "/workspace/libero_data"
os.makedirs("/workspace/libero_data", exist_ok=True)

from libero.libero import get_libero_path, benchmark
from libero.libero.envs import OffScreenRenderEnv

benchmark_dict = benchmark.get_benchmark_dict()
task_suite = benchmark_dict[TASK_SUITE_NAME]()
task = task_suite.get_task(TASK_ID)
task_description_libero = task.language
task_bddl_file = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)

env = OffScreenRenderEnv(bddl_file_name=task_bddl_file,
                         camera_heights=ENV_IMAGE_SIZE, camera_widths=ENV_IMAGE_SIZE)
env.seed(SEED)
init_states = task_suite.get_task_init_states(TASK_ID)
print(f"Task: {task_description_libero}")
print(f"Init states: {len(init_states)}")


`torch_dtype` is deprecated! Use `dtype` instead!


Loading  HuggingFaceTB/SmolVLM2-500M-Instruct weights ...


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

[robosuite WARNING] No private macro file found! (__init__.py:7)


[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)


[robosuite WARNING] To setup, run: python /venv/main/lib/python3.12/site-packages/robosuite/scripts/setup_macros.py (__init__.py:9)


[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Local assets not found. Downloading from HuggingFace Hub...
Assets already downloaded at /root/.cache/libero/assets


Task: pick up the black bowl between the plate and the ramekin and place it on the plate
Init states: 50


## 10. Q-select эвал helpers

In [11]:
def quat2axisangle(quat):
    quat = quat.astype(np.float32).copy()
    quat[3] = np.clip(quat[3], -1.0, 1.0)
    den = np.sqrt(max(1e-12, 1.0 - quat[3]**2))
    if np.isclose(den, 0.0): return np.zeros(3, dtype=np.float32)
    angle = 2.0 * math.acos(float(quat[3]))
    return ((quat[:3] * angle) / den).astype(np.float32)

def rotate_180(im): return np.ascontiguousarray(im[::-1, ::-1])


def libero_obs_to_critic_input(raw_obs):
    """raw LIBERO obs (256×256) → (img1, img2, state) для critic'а.
    GPU-side downscale через тот же режим что в обучении."""
    a_img = rotate_180(raw_obs["agentview_image"]).astype(np.uint8)
    w_img = rotate_180(raw_obs["robot0_eye_in_hand_image"]).astype(np.uint8)
    
    img1_full = torch.from_numpy(a_img).permute(2, 0, 1).unsqueeze(0).float().to(device) / 255.0
    img2_full = torch.from_numpy(w_img).permute(2, 0, 1).unsqueeze(0).float().to(device) / 255.0
    img1_t = downscale_only(img1_full, CRITIC_IMG_SIZE)
    img2_t = downscale_only(img2_full, CRITIC_IMG_SIZE)
    
    eef_pos = raw_obs["robot0_eef_pos"].astype(np.float32)
    eef_quat = raw_obs["robot0_eef_quat"].astype(np.float32)
    grip = raw_obs["robot0_gripper_qpos"].astype(np.float32)
    state = np.concatenate([eef_pos, quat2axisangle(eef_quat), grip], 0).astype(np.float32)
    state_norm = (state - state_mean) / state_std
    state_t = torch.from_numpy(state_norm).unsqueeze(0).to(device)
    
    return img1_t, img2_t, state_t


def libero_obs_to_lerobot(raw_obs, task_text):
    """Для policy SmolVLA — нужно полное разрешение и task token."""
    a_img = rotate_180(raw_obs["agentview_image"])
    w_img = rotate_180(raw_obs["robot0_eye_in_hand_image"])
    eef_pos = raw_obs["robot0_eef_pos"].astype(np.float32)
    eef_quat = raw_obs["robot0_eef_quat"].astype(np.float32)
    grip = raw_obs["robot0_gripper_qpos"].astype(np.float32)
    state = np.concatenate([eef_pos, quat2axisangle(eef_quat), grip], 0).astype(np.float32)
    return {
        "observation.images.image":  torch.from_numpy(a_img).permute(2,0,1).unsqueeze(0).float()/255.0,
        "observation.images.image2": torch.from_numpy(w_img).permute(2,0,1).unsqueeze(0).float()/255.0,
        "observation.state":         torch.from_numpy(state).unsqueeze(0).float(),
        "task":                      [task_text],
    }


@torch.no_grad()
def sample_chunks_with_q(raw_obs, task_text, n_candidates=8, num_steps_override=4):
    """Семплирует n_candidates chunks от baseline policy + scoring через critic."""
    from lerobot.policies.smolvla.modeling_smolvla import resize_with_pad, pad_vector, make_att_2d_masks
    
    obs = libero_obs_to_lerobot(raw_obs, task_text)
    obs = preprocessor(obs)
    
    img1_pol = obs["observation.images.image"].to(device)
    img2_pol = obs["observation.images.image2"].to(device)
    img1_pol = resize_with_pad(img1_pol, 512, 512, pad_value=0) * 2.0 - 1.0
    img2_pol = resize_with_pad(img2_pol, 512, 512, pad_value=0) * 2.0 - 1.0
    state_pol = pad_vector(obs["observation.state"].to(device), policy.config.max_state_dim)
    lang_tokens = obs["observation.language.tokens"].to(device)
    lang_masks  = obs["observation.language.attention_mask"].to(device)
    mask1 = torch.ones(1, dtype=torch.bool, device=device)
    mask2 = torch.ones(1, dtype=torch.bool, device=device)
    
    # Embed prefix
    prefix_embs, prefix_pad_masks, prefix_att_masks = policy.model.embed_prefix(
        [img1_pol, img2_pol], [mask1, mask2], lang_tokens, lang_masks, state=state_pol
    )
    prefix_att_2d  = make_att_2d_masks(prefix_pad_masks, prefix_att_masks)
    prefix_pos_ids = torch.cumsum(prefix_pad_masks, dim=1) - 1
    _, past_kv = policy.model.vlm_with_expert.forward(
        attention_mask=prefix_att_2d, position_ids=prefix_pos_ids,
        past_key_values=None, inputs_embeds=[prefix_embs, None],
        use_cache=True, fill_kv_cache=True,
    )
    
    actions_shape = (1, policy.config.chunk_size, policy.config.max_action_dim)
    num_steps = num_steps_override or policy.config.num_steps
    dt = -1.0 / num_steps
    
    # Также готовим images для critic'а
    img1_critic, img2_critic, state_critic = libero_obs_to_critic_input(raw_obs)
    
    chunks = []
    q_scores = []
    for _ in range(n_candidates):
        x_t = policy.model.sample_noise(actions_shape, device)
        for step in range(num_steps):
            t_val = 1.0 + step * dt
            t_tensor = torch.tensor(t_val, device=device).expand(1)
            v_t = policy.model.denoise_step(prefix_pad_masks, past_kv, x_t, t_tensor)
            x_t = x_t + dt * v_t
        chunks.append(x_t)
        # Scoring через critic — берём первый action из chunk'а в исходных 7 dim
        first_action = x_t[0, 0, :7].unsqueeze(0)
        q = critic.min_q(img1_critic, img2_critic, state_critic, first_action)
        q_scores.append(q.item())
    
    best_idx = int(np.argmax(q_scores))
    return chunks[best_idx][0], q_scores, best_idx


## 11. Q-select эвал на 13 init_states

In [12]:
@torch.no_grad()
def predict_q_select(raw_obs, task_text, queue=[]):
    if len(queue) == 0:
        best_chunk, q_scores, best_idx = sample_chunks_with_q(raw_obs, task_text)
        for t in range(policy.config.chunk_size):
            a_norm = best_chunk[t:t+1, :7]
            a_post = postprocessor(a_norm)
            queue.append(a_post.squeeze(0).cpu().numpy().astype(np.float32))
        return queue.pop(0), {"q_scores": q_scores, "best_idx": best_idx}
    return queue.pop(0), None


print("="*60)
print("Q-SELECT eval (ResNet critic)")
print("="*60)
n_success = 0
all_q = []
t0 = time.time()
for state_id in INIT_STATES_IDS:
    policy.reset()
    env.reset()
    raw_obs = env.set_init_state(init_states[state_id])
    queue = []
    success = False
    chunks = 0
    for t in range(MAX_STEPS + NUM_STABILIZATION_STEPS):
        if t < NUM_STABILIZATION_STEPS:
            action = [0.0]*6 + [-1.0]
        else:
            a_np, info = predict_q_select(raw_obs, task_description_libero, queue=queue)
            action = a_np.tolist()
            if info is not None:
                chunks += 1
                all_q.append(info["q_scores"])
        raw_obs, _, done, _ = env.step(action)
        if done:
            success = True
            break
    if success: n_success += 1
    print(f"  state {state_id:3d}: {'✓' if success else '✗'} ({t+1} steps, {chunks} chunks)")

elapsed = time.time() - t0
sr = n_success / len(INIT_STATES_IDS)
print(f"\nQ-SELECT (ResNet critic) SR: {n_success}/{len(INIT_STATES_IDS)} = {sr:.2%}")
print(f"Wall time: {elapsed/60:.1f} min")

if all_q:
    arr = np.array(all_q)
    print(f"\n── Q-scores stats ──")
    print(f"  total chunk-replans: {len(all_q)}")
    print(f"  mean Q:               {arr.mean():.4f}")
    print(f"  std within batch avg: {arr.std(axis=1).mean():.4f}")
    print(f"  best - mean avg:      {(arr.max(axis=1)-arr.mean(axis=1)).mean():.4f}")
    print(f"  best - worst avg:     {(arr.max(axis=1)-arr.min(axis=1)).mean():.4f}")


Q-SELECT eval (ResNet critic)


  state   1: ✗ (100 steps, 2 chunks)


  state   2: ✗ (100 steps, 2 chunks)


  state   6: ✓ (78 steps, 2 chunks)


  state   7: ✗ (100 steps, 2 chunks)


...

  state  32: ✗ (100 steps, 2 chunks)


  state  35: ✓ (99 steps, 2 chunks)


  state  38: ✗ (100 steps, 2 chunks)


  state  47: ✗ (100 steps, 2 chunks)


  state  46: ✓ (78 steps, 2 chunks)

Q-SELECT (ResNet critic) SR: 5/13 = 38.46%
Wall time: 1.3 min

── Q-scores stats ──
  total chunk-replans: 26
  mean Q:               -39.1300
  std within batch avg: 0.7039
  best - mean avg:      0.9428
  best - worst avg:     2.1877


## 12. Вердикт

In [13]:
print("="*60)
print("РЕЗУЛЬТАТ ResNet critic vs предыдущие")
print("="*60)
print(f"  Baseline SmolVLA (no critic):     ~7-9/13")
print(f"  Q-select v1 (mean prefix VLM):     8/13 (предыдущие тесты)")
print(f"  Q-select ResNet critic:            {n_success}/13 = {sr:.2%}")
print()
if n_success > 9:
    print("✓ ResNet critic РАБОТАЕТ! Раньше проблема была в эмбеддингах от VLM.")
    print("  → Можно делать distillation policy с этим critic'ом.")
elif n_success >= 8:
    print("○ ResNet critic ≈ baseline. Эмбеддинги были не хуже.")
    print("  → Проблема глубже — возможно сам PA-RL подход не работает на этой задаче.")
else:
    print("⚠ ResNet critic ХУЖЕ baseline. Эмбеддинги не были проблемой.")
    print("  → Принципиальная проблема: critic + 50k frames + sparse reward не дают signal.")


РЕЗУЛЬТАТ ResNet critic vs предыдущие
  Baseline SmolVLA (no critic):     ~7-9/13
  Q-select v1 (mean prefix VLM):     8/13 (предыдущие тесты)
  Q-select ResNet critic:            5/13 = 38.46%

⚠ ResNet critic ХУЖЕ baseline. Эмбеддинги не были проблемой.
  → Принципиальная проблема: critic + 50k frames + sparse reward не дают signal.
